# Python Fundamentals: NumPy Arrays vs Lists
## Day 4 - Notebook 15
***
This notebook covers:
- Why NumPy arrays are better for numerical calculations than Python lists
- Bit-by-bit explanation: memory, types, vectorization
- When to use lists vs arrays
***


## 1 Python Lists: How They Work

A Python list is a **dynamic array of references** to Python objects. Each element can be any type (int, float, string, another list). The list stores *pointers* to those objects, not the raw data itself. That flexibility is great for mixed data, but costly for pure numbers.


In [ ]:
# Lists: flexible but costly for numbers
lst = [1, 2, 3, 4, 5]
print(lst)
# Each element is a Python int object (with overhead)
# List stores pointers to these objects

## 2 NumPy Arrays: Homogeneous and Contiguous

A NumPy array stores **one fixed data type** in a **contiguous block of memory**. No Python object overhead per element—just raw bytes. Example: an array of 1000 floats uses ~8KB for data plus small array metadata, while a list of 1000 floats stores 1000 pointers plus 1000 Python float objects.


In [ ]:
import numpy as np

arr = np.array([1, 2, 3, 4, 5], dtype=np.float64)
print(arr)
print("dtype:", arr.dtype)
print("itemsize (bytes per element):", arr.itemsize)
# All elements stored sequentially in memory

### 2.1 Memory Layout: Pointers vs Raw Data

| Aspect | Python list | NumPy array |
|--------|-------------|-------------|
| Storage | Pointers to objects | Raw numeric data |
| Types | Mixed allowed | Single dtype |
| Memory | Higher (overhead per element) | Lower (compact) |
| Access | Indirect (follow pointer) | Direct (index into block) |

For large numerical datasets, NumPy uses much less memory and allows faster access because the CPU can work with contiguous memory and predictable types.

### 2.2 Overhead in Bits: Definitive Numbers

**NumPy (pure data, no per-element overhead):**
| dtype | Bits per element |
|-------|------------------|
| np.int8 | 8 bits |
| np.int16 | 16 bits |
| np.int32 | 32 bits |
| np.int64 | 64 bits |
| np.float32 | 32 bits |
| np.float64 | 64 bits |

**Python list of integers (per element):**
- **Pointer** in the list: 64 bits (8 bytes on 64-bit systems)
- **Python int object**: ~224 bits (28 bytes) for a small integer—reference count, type pointer, size field, and the digit array. The actual numeric value may need only 16–64 bits; the rest is overhead.

So for one small integer in a list: **64 + 224 = 288 bits** minimum, vs **16 bits** for np.int16 or **64 bits** for np.int64. Python uses roughly **4–18× more bits** per integer.

**Example for 1000 integers:**
- List: 1000 × (64 bits pointer + 224 bits int) ≈ **288,000 bits** (~36 KB)
- np.int64 array: 1000 × 64 bits = **64,000 bits** (8 KB)
- np.int16 array: 1000 × 16 bits = **16,000 bits** (2 KB)


In [ ]:
# Definitive bit/byte comparison (Python 3, 64-bit)
import sys

# Per-element overhead
py_int = 42
bits_per_pointer = 64  # on 64-bit systems
print("Python int object:", sys.getsizeof(py_int), "bytes =", sys.getsizeof(py_int) * 8, "bits")
print("List pointer (64-bit):", 8, "bytes =", bits_per_pointer, "bits")
print()

# NumPy dtypes (bits per element)
for dtype, name in [(np.int8, "int8"), (np.int16, "int16"), (np.int64, "int64"), (np.float64, "float64")]:
    a = np.array([1], dtype=dtype)
    print(f"np.{name}: {a.itemsize} bytes = {a.itemsize * 8} bits per element")
print()

# 1000 elements
n = 1000
lst = list(range(n))
arr_i64 = np.arange(n, dtype=np.int64)
arr_i16 = np.arange(n, dtype=np.int16)

list_bytes = sys.getsizeof(lst) + sum(sys.getsizeof(x) for x in lst)
print(f"List of {n} ints: ~{list_bytes:,} bytes = ~{list_bytes * 8:,} bits")
print(f"NumPy int64, {n} elements: {arr_i64.nbytes:,} bytes = {arr_i64.nbytes * 8:,} bits")
print(f"NumPy int16, {n} elements: {arr_i16.nbytes:,} bytes = {arr_i16.nbytes * 8:,} bits")

## 3 Vectorization: One Operation, Many Elements

With a **list**, an operation like `[x*2 for x in lst]` runs a Python loop: each element is fetched, multiplied, and a new object is created. With a **NumPy array**, `arr * 2` is a single **vectorized** operation: the work is done in optimized C/Fortran code, no Python loop. This is why NumPy is much faster for bulk numerical work.


In [ ]:
# List: Python loop (slow)
lst = list(range(1000000))
result_list = [x * 2 for x in lst]

# NumPy: vectorized (fast)
arr = np.arange(1000000)
result_arr = arr * 2

print("Both give same result (first 5):", result_list[:5], result_arr[:5].tolist())

In [ ]:
# Speed comparison (optional: run if interested)
import time

n = 1_000_000
lst = list(range(n))
arr = np.arange(n)

t0 = time.perf_counter()
_ = [x * 2 for x in lst]
t_list = time.perf_counter() - t0

t0 = time.perf_counter()
_ = arr * 2
t_arr = time.perf_counter() - t0

print(f"List: {t_list:.4f} s")
print(f"NumPy: {t_arr:.4f} s")
print(f"NumPy is ~{t_list/t_arr:.0f}x faster")

## 4 Built-in Numerical Functions

NumPy provides **mean**, **sum**, **std**, **min**, **max** etc. as methods or `np.` functions. These run in C, avoiding Python loops. With lists you would need a loop or the `statistics` module, which is still slower for large data.


In [ ]:
arr = np.array([3.1, 4.2, 5.3, 6.4, 7.5])
print("mean:", arr.mean())
print("sum:", arr.sum())
print("std:", arr.std())
print("min/max:", arr.min(), arr.max())

## 5 When to Use What?

| Use lists when | Use NumPy when |
|-----------------|----------------|
| Mixed types (strings, numbers, objects) | Homogeneous numerical data |
| Dynamic appends, variable length | Fixed-size arrays, matrices |
| Small collections, general purpose | Large datasets, math, plots |
| Need Python list methods | Need vectorized math, linear algebra |

For scientific computing, simulations, and data analysis, NumPy arrays are the standard choice.


#### 5.1 Tasks:

> (a) Create a NumPy array from the list [2, 4, 6, 8, 10] and print its mean, sum, and max. <br>
> (b) Create an array with np.linspace(0, 1, 11). Print it. What is the shape? <br>
> (c) Create an array of 100 zeros using np.zeros(100). Add 5 to every element (vectorized) and print the first 5 elements. <br>
> (d) Create a NumPy array with values 1 to 20 (np.arange). Compute the mean using the array's .mean() method and print it.

In [ ]:
# Your solution:

#### Solution:

In [ ]:
# Model solution (a)
import numpy as np
arr = np.array([2, 4, 6, 8, 10])
print(arr.mean(), arr.sum(), arr.max())

# Model solution (b)
arr = np.linspace(0, 1, 11)
print(arr)
print("Shape:", arr.shape)

# Model solution (c)
arr = np.zeros(100)
arr = arr + 5
print(arr[:5])

# Model solution (d)
arr = np.arange(1, 21)
print(arr.mean())